### 1. Import Libraries

In [98]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


### 2. Read CSV Files

In [99]:
ges_degree_details = pd.read_csv("Data/ges_degree_details.csv")
ges_employment_outcomes = pd.read_csv("Data/ges_employment_outcomes.csv")
employment_by_industry = pd.read_csv("Data/employment_by_industry.csv")
retrenchment_by_industry = pd.read_csv("Data/retrenchment_by_industry.csv")
job_vacancies_by_industry = pd.read_csv("Data/job_vacancies_by_industry.csv")




### 3. Show First 5 Rows Of Datasets

In [100]:
ges_degree_details.head()

,year,university,school,degree,Key
0,2013,Nanyang Technological University,College of Business (Nanyang Business School),Accountancy and Business,1
1,2013,Nanyang Technological University,College of Business (Nanyang Business School),Accountancy (3-yr direct Honours Programme),2
2,2013,Nanyang Technological University,College of Business (Nanyang Business School),Business (3-yr direct Honours Programme),3
3,2013,Nanyang Technological University,College of Business (Nanyang Business School),Business and Computing,4
4,2013,Nanyang Technological University,College of Engineering,Aerospace Engineering,5


In [101]:
ges_employment_outcomes.head()

,employment_rate_overall,employment_rate_ft_perm,basic_monthly_mean,basic_monthly_median,gross_monthly_mean,gross_monthly_median,gross_mthly_25_percentile,gross_mthly_75_percentile,gross_monthly_mean.1,gross_monthly_median.1,...,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Key
0,97.4,96.1,3701,3200,3727,3350,2900,4000,3727,3350,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,97.1,95.7,2850,2700,2938,2700,2700,2900,2938,2700,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
2,90.9,85.7,3053,3000,3214,3000,2700,3500,3214,3000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
3,87.5,87.5,3557,3400,3615,3400,3000,4100,3615,3400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
4,95.3,95.3,3494,3500,3536,3500,3100,3816,3536,3500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5


In [102]:
employment_by_industry.head()

,year,sex,industry1,industry2,employment_status,employed
0,2010,male,manufacturing,manufacturing,employers,10800
1,2010,male,manufacturing,manufacturing,employees,170500
2,2010,male,manufacturing,manufacturing,own account workers,4400
3,2010,male,manufacturing,manufacturing,contributing family workers,400
4,2010,male,construction,construction,employers,11000


In [103]:
retrenchment_by_industry.head()

,year,industry,incidence_of_retrenchment
0,1998,manufacturing,62.9
1,1998,"food, beverages and tobacco",15.4
2,1998,textile and wearing apparel,31.1
3,1998,paper products and publishing,27
4,1998,petroleum and chemical products,30.1


In [104]:
job_vacancies_by_industry.head()

,year,industry,occupation,job_vacancy_rate
0,1998,manufacturing,"professionals, managers, executives and techni...",1.8
1,1998,manufacturing,"clerical, sales and services workers",1.2
2,1998,manufacturing,"production and transport operators, cleaners a...",1.6
3,1998,"food, beverages and tobacco","professionals, managers, executives and techni...",-
4,1998,"food, beverages and tobacco","clerical, sales and services workers",1.3


### 4. Data Cleaning

#### 4.1. Code to make a copy of ges_degree_details dataframe for cleaning

In [105]:
ges_degree_details_clean = ges_degree_details.copy()
print("Shape:", ges_degree_details_clean.shape)

Shape: (5135, 5)


### 4.2. Code to check data quality

In [106]:
column_summary = pd.DataFrame({
    "data_type": ges_degree_details_clean.dtypes.astype(str),
    "missing_count": (
        ges_degree_details_clean.isna().sum()
        ),

    "missing_percentage": (
        ges_degree_details_clean.isna().mean() * 100
    ).round(2),
    "unique_count": (
        ges_degree_details_clean.nunique(dropna=False)
    )
})

display(column_summary)

print("Empty Rows:", ges_degree_details_clean.isna().all(axis=1).sum())
print("Exact duplicate rows:", ges_degree_details_clean.duplicated().sum())
print("Duplicate keys:", ges_degree_details_clean["Key"].duplicated().sum())

,data_type,missing_count,missing_percentage,unique_count
year,str,0,0.0,12
university,str,0,0.0,7
school,str,0,0.0,71
degree,str,0,0.0,349
Key,int64,0,0.0,5135


Empty Rows: 0
Exact duplicate rows: 0
Duplicate keys: 0


### 4.3. Code to identify placeholder values 

In [107]:
placeholder_values = {
    "",
    "na",
    "n/a",
    "null",
    "none"
}

placeholder_report = []

for column in ["university", "school", "degree"]:
    normalised_values = (
        ges_degree_details_clean[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    counts = normalised_values[
        normalised_values.isin(placeholder_values)
        ].value_counts()
    
    for value, count in counts.items():
        placeholder_report.append({
            "column": column,
            "placeholder_value": value,
            "count": count
        })

placeholder_report = pd.DataFrame(placeholder_report)

display(placeholder_report)

,column,placeholder_value,count
0,school,na,128


### 4.4. Code to convert placeholders to missing values

In [108]:
for column in ["university", "school", "degree"]:
    normalised_values = (
        ges_degree_details_clean[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    placeholder_mask = normalised_values.isin(placeholder_values)

    ges_degree_details_clean.loc[
        placeholder_mask,
        column 
    ] = pd.NA

In [109]:
text_columns = [
    "university",
    "school",
    "degree"
]

for column in text_columns:
    ges_degree_details_clean[column] = (
        ges_degree_details_clean[column].astype("string").str.strip()
    )


### 4.5. Code to verify missing values

In [110]:
print("Missing values after converting placeholders:")
display(ges_degree_details_clean.isna().sum())

Missing values after converting placeholders:


year            0
university      0
school        128
degree          0
Key             0
dtype: int64

In [111]:
missing_school_by_university = (
    ges_degree_details_clean.loc[
        ges_degree_details_clean["school"].isna(), "university"

    ].value_counts(dropna=False)
)

display(missing_school_by_university)

university
Singapore University of Technology and Design    128
Name: count, dtype: int64[pyarrow]

### 4.6. Code to inspect and validate year column

In [112]:
year_counts = (
    ges_degree_details_clean["year"].value_counts(dropna=False).sort_index()
)
display(year_counts)

year
2012     86
2013    340
2014    436
2015    492
2016    484
2017    528
2018    548
2019    560
2020    556
2021    540
2022    564
year      1
Name: count, dtype: int64

In [113]:
year_numeric_test = pd.to_numeric(
    ges_degree_details_clean["year"],
    errors="coerce"
)

### 4.7. Code to remove invalid year 

In [114]:
invalid_year_mask = (
    year_numeric_test.isna()
    & ges_degree_details_clean["year"].notna()
)

print("Rows containing invalid year values:", invalid_year_mask.sum())

display(ges_degree_details_clean.loc[
    invalid_year_mask
])

Rows containing invalid year values: 1


,year,university,school,degree,Key
1262,year,university,school,degree,1263


In [ ]:
rows_before = len(ges_degree_details_clean)

ges_degree_details_clean = (
    ges_degree_details_clean
    .loc[~invalid_year_mask]
    .copy()
    .reset_index(drop=True)
)

rows_removed = (
    rows_before - len(ges_degree_details_clean)
)

print("Rows removed due to invalid year values:", rows_removed)

Rows removed due to invalid year values: 1


In [116]:
ges_degree_details_clean["year"] = pd.to_numeric(
    ges_degree_details_clean["year"],
    errors = "coerce"
).astype("Int64")

In [117]:
print(
    ges_degree_details_clean["year"].dtype
)

display(
    ges_degree_details_clean["year"].value_counts(dropna=False).sort_index()
)

Int64


year
2012     86
2013    340
2014    436
2015    492
2016    484
2017    528
2018    548
2019    560
2020    556
2021    540
2022    564
Name: count, dtype: Int64

### 4.8. Code to inspect university and school names

In [118]:
school_counts = (
    ges_degree_details_clean["school"].value_counts(dropna=False)
)

display(school_counts)

school
College of Engineering                                       669
College of Humanities, Arts & Social Sciences                376
Faculty of Engineering                                       274
Faculty of Science                                           222
School of Computing                                          218
                                                            ... 
Singapore Institute of Technology -Trinity College Dublin      4
YST Conservatory Of Music                                      4
SIT-DigiPen Institute of Technology                            4
SIT-Massey University                                          4
School of Law                                                  4
Name: count, Length: 70, dtype: int64[pyarrow]

In [119]:
school_name_corrections = {
    "Faculty Of Dentistry": "Faculty of Dentistry",
    "Faculty Of Engineering": "Faculty of Engineering",
    "YST Conservatory Of Music": "YST Conservatory of Music"
}

ges_degree_details_clean["school"] = (
    ges_degree_details_clean["school"].replace(school_name_corrections)
)

In [120]:
display(
    ges_degree_details_clean["school"].value_counts(dropna=False)
)

school
College of Engineering                                       669
College of Humanities, Arts & Social Sciences                376
Faculty of Engineering                                       350
Faculty of Science                                           222
School of Computing                                          218
                                                            ... 
SIT-Trinity College Dublin                                     8
Singapore Institute of Technology -Trinity College Dublin      4
SIT-DigiPen Institute of Technology                            4
SIT-Massey University                                          4
School of Law                                                  4
Name: count, Length: 67, dtype: int64[pyarrow]

In [121]:
display(
    ges_degree_details_clean["university"].value_counts(dropna=False)
)

university
Nanyang Technological University                 1627
National University of Singapore                 1627
Singapore Institute of Technology                1136
Singapore Management University                   492
Singapore University of Technology and Design     128
Singapore University of Social Sciences           124
Name: count, dtype: int64[pyarrow]

In [122]:
degree_counts = (
    ges_degree_details_clean["degree"].value_counts(dropna=False)
)

display(degree_counts)

degree
Economics                                                         69
Accountancy                                                       56
Bachelor of Arts                                                  41
Bachelor of Arts (Hons)                                           41
Bachelor of Social Sciences                                       41
                                                                  ..
Biological Sciences and Psychology ^                               4
Bachelor of Law **                                                 4
Bachelor of Science (Air Transport Management)                     4
Bachelor of Engineering with Honours in Aerospace Engineering      4
Bachelor of Engineering with Honours in Mechanical Engineering     4
Name: count, Length: 348, dtype: int64[pyarrow]

In [123]:
degree_case_check = pd.DataFrame()

degree_case_check["degree"] = (
    ges_degree_details_clean["degree"]
)

degree_case_check["degree_lowercase"] = (
    ges_degree_details_clean["degree"].str.lower()
)

degree_case_check = (
    degree_case_check.dropna().drop_duplicates()
)

degree_case_check = degree_case_check[degree_case_check["degree_lowercase"].duplicated(keep=False)]

display (
    degree_case_check.sort_values("degree_lowercase")
)

,degree,degree_lowercase
33,Arts (with Education),arts (with education)
610,Arts (With Education),arts (with education)
381,Bachelor of Computing (Communications And Medi...,bachelor of computing (communications and medi...
637,Bachelor of Computing (Communications and Medi...,bachelor of computing (communications and medi...
61,Bachelor of Engineering (Materials Science and...,bachelor of engineering (materials science and...
368,Bachelor of Engineering (Materials Science And...,bachelor of engineering (materials science and...
148,Bachelor of Medicine and Bachelor of Surgery #,bachelor of medicine and bachelor of surgery #
389,Bachelor of Medicine and Bachelor Of Surgery #,bachelor of medicine and bachelor of surgery #
516,Bachelor of Medicine And Bachelor Of Surgery #,bachelor of medicine and bachelor of surgery #
6,Chemical and Biomolecular Engineering,chemical and biomolecular engineering


In [124]:
degree_name_corrections = {
    "Arts (With Education)":
        "Arts (with Education)",

    "Bachelor of Computing (Communications And Media) **":
        "Bachelor of Computing (Communications and Media) **",

    "Bachelor of Engineering (Materials Science And Engineering)":
        "Bachelor of Engineering (Materials Science and Engineering)",

    "Bachelor of Medicine and Bachelor Of Surgery #":
        "Bachelor of Medicine and Bachelor of Surgery #",

    "Bachelor of Medicine And Bachelor Of Surgery #":
        "Bachelor of Medicine and Bachelor of Surgery #",
    
    "Chemical And Biomolecular Engineering":
        "Chemical and Biomolecular Engineering",
    
    "Chemical And Biomolecular Engineering and Economics **":
        "Chemical and Biomolecular Engineering and Economics **",
    
    "Electrical And Electronic Engineering":
        "Electrical and Electronic Engineering",

    "Information Engineering And Media":
        "Information Engineering and Media",
    
    "Linguistics And Multilingual Studies":
        "Linguistics and Multilingual Studies",

    "Public Policy And Global Affairs":
        "Public Policy and Global Affairs",
    
    "Science (With Education)":
        "Science (with Education)"
}

ges_degree_details_clean["degree"] = (
    ges_degree_details_clean["degree"].replace(degree_name_corrections)
)

In [125]:
print(
    "Unique Degree Names: ", ges_degree_details_clean["degree"].nunique()
)

Unique Degree Names:  336


In [126]:
programme_year_columns = [
    "year",
    "university",
    "school",
    "degree"
]

logical_duplicate_mask = (
    ges_degree_details_clean.duplicated(subset = programme_year_columns, keep=False)
)

logical_duplicate_excess = (
    ges_degree_details_clean.duplicated(subset=programme_year_columns, keep="first").sum()
)

print("Rows involved in repeated programme-year groups:", logical_duplicate_mask.sum())
print("Additional repeated records beyond the first:", logical_duplicate_excess)

Rows involved in repeated programme-year groups: 5050
Additional repeated records beyond the first: 3787


In [127]:
logical_duplicate_group_sizes = (
    ges_degree_details_clean.loc[logical_duplicate_mask].groupby(programme_year_columns, dropna=False).size().reset_index(name="record_count")
)

print("Frequency of repeated group sizes")

display(
    logical_duplicate_group_sizes["record_count"].value_counts().sort_index().rename_axis("number_of_copies").reset_index(name="number_of_programme_year_groups")
)

Frequency of repeated group sizes


,number_of_copies,number_of_programme_year_groups
0,2,1
1,4,1262


In [128]:
logical_duplicate_rows = (
    ges_degree_details_clean.loc[logical_duplicate_mask].sort_values(programme_year_columns + ["Key"])
)

display(
    logical_duplicate_rows.head(20)
)

,year,university,school,degree,Key
5053,2012,Nanyang Technological University,College of Engineering,Aerospace Engineering,5054
5095,2012,Nanyang Technological University,College of Engineering,Aerospace Engineering,5096
1,2013,Nanyang Technological University,College of Business (Nanyang Business School),Accountancy (3-yr direct Honours Programme),2
1264,2013,Nanyang Technological University,College of Business (Nanyang Business School),Accountancy (3-yr direct Honours Programme),1265
2526,2013,Nanyang Technological University,College of Business (Nanyang Business School),Accountancy (3-yr direct Honours Programme),2527
3788,2013,Nanyang Technological University,College of Business (Nanyang Business School),Accountancy (3-yr direct Honours Programme),3789
0,2013,Nanyang Technological University,College of Business (Nanyang Business School),Accountancy and Business,1
1263,2013,Nanyang Technological University,College of Business (Nanyang Business School),Accountancy and Business,1264
2525,2013,Nanyang Technological University,College of Business (Nanyang Business School),Accountancy and Business,2526
3787,2013,Nanyang Technological University,College of Business (Nanyang Business School),Accountancy and Business,3788


In [ ]:
print("FINAL DATA-QUALITY SUMMARY")
print("-" * 40)

print(
    "Final shape:",
    ges_degree_details_clean.shape
)

print("\nMissing values:")
print(
    ges_degree_details_clean
    .isna()
    .sum()
)

print("\nData types:")
print(
    ges_degree_details_clean.dtypes
)

print(
    "\nExact duplicate rows:",
    ges_degree_details_clean
    .duplicated()
    .sum()
)

print(
    "Duplicate Key values:",
    ges_degree_details_clean["Key"]
    .duplicated()
    .sum()
)

print(
    "Additional repeated programme-year records:",
    logical_duplicate_excess
)

Final shape: (5134, 5)

Missing values:
year            0
university      0
school        128
degree          0
Key             0
dtype: int64

Data types:
year           Int64
university    string
school        string
degree        string
Key            int64
dtype: object

Exact duplicate rows: 0
Duplicate keys: 0


## 5. Employment Outcomes Cleaning (Tahmin)

Cleaning `ges_employment_outcomes.csv`, which will be merged with the cleaned `ges_degree_details_clean` above on the shared `Key` column.

In [ ]:
ges_employment_outcomes_clean = ges_employment_outcomes.copy()
print("Shape:", ges_employment_outcomes_clean.shape)
ges_employment_outcomes_clean.head()

## 2. Initial inspection

Checking structure, dtypes, and nulls before doing anything else.

In [ ]:
ges_employment_outcomes_clean.info()

In [ ]:
# How many columns are entirely empty?
empty_cols = ges_employment_outcomes_clean.columns[ges_employment_outcomes_clean.isna().all()]
print(f"Fully empty columns: {len(empty_cols)}")
print(list(empty_cols))

**Finding:** The raw CSV has 48 columns, but only 12 contain any data. The other columns (mostly `Unnamed: 12` through `Unnamed: 46`) are artifacts of trailing commas in the source file and are completely empty. These will be dropped in Step 4.

## 3. Remove embedded duplicate header row(s)

This dataset appears to have been assembled by stacking multiple yearly tables. Occasionally, a header row from one of the original tables got pulled in as a data row. We detect this by checking whether a row's value equals its own column name.

In [ ]:
# A genuine data row should never contain the literal column name as its value.
# Flag any row where the first data column's value equals its own column header.
header_mask = ges_employment_outcomes_clean["employment_rate_overall"] == "employment_rate_overall"
print(f"Embedded header rows found: {header_mask.sum()}")
ges_employment_outcomes_clean[header_mask]

In [ ]:
ges_employment_outcomes_clean = ges_employment_outcomes_clean[~header_mask].reset_index(drop=True)
print("Shape after removing embedded header row(s):", ges_employment_outcomes_clean.shape)

## 4. Drop empty and duplicate columns

- The `Unnamed: *` columns are completely empty (100% null) — dropped.
- `gross_monthly_mean.1`, `gross_monthly_median.1`, `gross_mthly_25_percentile.1`, and `gross_mthly_75_percentile.1` are exact duplicates of their non-`.1` counterparts (verified below) — dropped.

In [ ]:
# Confirm the .1 columns are true duplicates before dropping them
dup_pairs = [
    ("gross_monthly_mean", "gross_monthly_mean.1"),
    ("gross_monthly_median", "gross_monthly_median.1"),
    ("gross_mthly_25_percentile", "gross_mthly_25_percentile.1"),
    ("gross_mthly_75_percentile", "gross_mthly_75_percentile.1"),
]

for a, b in dup_pairs:
    match_rate = (ges_employment_outcomes_clean[a].fillna("MISSING") == ges_employment_outcomes_clean[b].fillna("MISSING")).mean()
    print(f"{a} vs {b}: {match_rate:.2%} identical")

In [ ]:
# Drop fully empty columns + confirmed duplicate columns
cols_to_drop = list(empty_cols) + [b for _, b in dup_pairs]
ges_employment_outcomes_clean = ges_employment_outcomes_clean.drop(columns=cols_to_drop)

print("Shape after dropping empty/duplicate columns:", ges_employment_outcomes_clean.shape)
ges_employment_outcomes_clean.head()

## 5. Standardise missing value markers

Missing values in this dataset are stored as the literal string `'na'` rather than a true null. This is why every numeric column was read in as text (`object`) dtype instead of numeric. We replace `'na'` with proper `NaN`.

In [ ]:
outcome_cols = [
    "employment_rate_overall",
    "employment_rate_ft_perm",
    "basic_monthly_mean",
    "basic_monthly_median",
    "gross_monthly_mean",
    "gross_monthly_median",
    "gross_mthly_25_percentile",
    "gross_mthly_75_percentile",
]

ges_employment_outcomes_clean[outcome_cols] = ges_employment_outcomes_clean[outcome_cols].replace("na", np.nan)

## 6. Convert columns to correct numeric dtype

Now that `'na'` has been replaced with real `NaN`, these columns can be safely converted to numeric types.

In [ ]:
for col in outcome_cols:
    ges_employment_outcomes_clean[col] = pd.to_numeric(ges_employment_outcomes_clean[col], errors="coerce")

ges_employment_outcomes_clean[outcome_cols].dtypes

In [ ]:
ges_employment_outcomes_clean.info()

## 7. Check for duplicate rows

`Key` should be unique (it's the join key to `ges_degree_details`), so we check both full-row duplicates and duplicate Keys.

In [ ]:
print("Fully duplicated rows:", ges_employment_outcomes_clean.duplicated().sum())
print("Duplicate Key values:", ges_employment_outcomes_clean['Key'].duplicated().sum())

## 8. Handle missing values

Each of the 8 outcome columns has the same number of missing values. This is expected: SkillsFuture/GES suppresses salary and employment rate figures for degree cohorts with very small graduating numbers (to protect individual privacy), rather than these being data entry errors.

**Decision:** We keep these rows with `NaN` rather than dropping them, for two reasons:
1. Dropping rows here would break the row-alignment with `ges_degree_details.csv` on `Key`, since that dataset has no missing rows to match against.
2. The missingness itself is informative (small/niche cohorts) — imputing a salary figure would misrepresent those cohorts. Rows with missing values can be filtered out later, at analysis time, only for the specific columns/visualisations that need complete data (e.g. `.dropna(subset=[...])` before a boxplot or KDE plot).

In [ ]:
missing_summary = ges_employment_outcomes_clean[outcome_cols].isna().sum().to_frame("missing_count")
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(ges_employment_outcomes_clean) * 100).round(2)
missing_summary

## 9. Outlier check (boxplot)

A quick boxplot of the salary columns to see the spread and flag any extreme outliers before this data gets used downstream. We don't remove outliers here — legitimately high or low salaries are real and relevant to the research question — but it's worth visualising and noting.

In [ ]:
salary_cols = ["basic_monthly_mean", "basic_monthly_median", "gross_monthly_mean", "gross_monthly_median"]

plt.figure(figsize=(10, 6))
sns.boxplot(data=ges_employment_outcomes_clean[salary_cols].dropna())
plt.title("Distribution of Monthly Salary Figures (Cleaned Data)")
plt.ylabel("S$ per month")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 10. Additional edge-case validation (Employment Outcomes)

Checking whether the cleaned values themselves are logically consistent, beyond just fixing structure and dtypes.

### 10.1 Logical consistency: does gross pay ever fall below basic pay?

Gross monthly salary is defined (per the assignment's metadata) as basic salary *plus* overtime, commissions, and allowances. It should never be lower than basic salary alone. We check for violations rather than assuming the relationship holds.

In [ ]:
basic_gt_gross_mean = ges_employment_outcomes_clean[ges_employment_outcomes_clean["basic_monthly_mean"] > ges_employment_outcomes_clean["gross_monthly_mean"]]
basic_gt_gross_median = ges_employment_outcomes_clean[ges_employment_outcomes_clean["basic_monthly_median"] > ges_employment_outcomes_clean["gross_monthly_median"]]

print(f"Rows where basic_monthly_mean > gross_monthly_mean: {len(basic_gt_gross_mean)}")
print(f"Rows where basic_monthly_median > gross_monthly_median: {len(basic_gt_gross_median)}")
basic_gt_gross_mean[["Key", "basic_monthly_mean", "gross_monthly_mean"]].head()

**Finding:** 21 rows have `basic_monthly_mean` slightly higher than `gross_monthly_mean` (and 4 rows for the median pair), by amounts ranging from about S$1 to S$30. This is too small and inconsistent to be a data entry error — it's most likely independent rounding in how MOE/universities computed each statistic, since the two figures come from separate survey questions rather than one being derived from the other. **Decision:** left as-is and documented, not "corrected," since we have no authoritative source to correct against and the discrepancy is immaterial to the analysis.

### 10.2 Logical consistency: percentile ordering

The 25th percentile should never exceed the median, and the median should never exceed the 75th percentile, for the same salary distribution.

In [ ]:
percentile_violations = ges_employment_outcomes_clean[
    (ges_employment_outcomes_clean["gross_mthly_25_percentile"] > ges_employment_outcomes_clean["gross_monthly_median"]) |
    (ges_employment_outcomes_clean["gross_monthly_median"] > ges_employment_outcomes_clean["gross_mthly_75_percentile"])
]
print(f"Percentile ordering violations: {len(percentile_violations)}")

**Finding:** 0 violations — the percentile columns are internally consistent. This is a good sign for the reliability of the salary figures overall.

### 10.3 Quantifying outliers with IQR (beyond the visual boxplot)

The boxplot in Step 9 shows outliers visually, but for the write-up it's more defensible to state how many there are and what the bounds are. Using the standard 1.5×IQR rule:

In [ ]:
def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((series < lower) | (series > upper)).sum()
    return lower, upper, n_outliers

for col in ["basic_monthly_mean", "basic_monthly_median", "gross_monthly_mean", "gross_monthly_median"]:
    lower, upper, n = iqr_outlier_count(ges_employment_outcomes_clean[col].dropna())
    print(f"{col}: bounds=({lower:.0f}, {upper:.0f}), outliers={n}")

**Decision:** These outliers are kept, not removed. High-earning outliers (e.g. medicine, law) and low-earning outliers (e.g. small arts/music cohorts) are real, legitimate data points that matter for the research question — removing them would bias the analysis toward "typical" degrees and away from exactly the conflicting high-salary-vs-stability comparisons the assignment brief asks us to investigate.

### 10.5 Sanity check: implausibly low "full-time permanent" salaries

`basic_monthly_mean` is defined as pay for *full-time permanent* work of at least 35 hours/week. A handful of values are unexpectedly low for full-time work — worth flagging even though we don't have grounds to remove them (this is official published data, not something we can independently correct).

In [ ]:
low_salary = ges_employment_outcomes_clean[ges_employment_outcomes_clean["basic_monthly_mean"] < 1000]
print(f"Rows with basic_monthly_mean under S$1,000: {len(low_salary)}")
low_salary[["Key", "basic_monthly_mean", "gross_monthly_mean", "employment_rate_overall"]].head(10)

> **Note:** the boxplot (Section 9) and IQR check (10.3) above run on `ges_employment_outcomes_clean` — before the merge, before excluding the corrupted `year == 2012` rows, and before resolving duplicate cohorts. They're kept here because they're what originally flagged that something was off (motivating the Section 8 investigation), but the counts here are inflated by rows that don't belong in the final dataset. The outlier check that should actually be reported/cited is Section 8.5, which runs on the final cleaned data.

**Finding:** 78 rows report a `basic_monthly_mean` under S$1,000. This is not a niche-cohort issue — every one of these rows turns out to be from `year == 2012`. Section 8 investigates this properly once `year` is available after merging, and finds that 2012's records are not reliable independent survey data at all (they are near-duplicates of 2013's figures, off by a scale factor) — so they are excluded from the final dataset rather than "corrected."

### 10.4 Outcome-only duplicate check — removed

An earlier version of this notebook checked how many rows shared an identical 8-value outcome tuple, using only the columns available in `ges_employment_outcomes.csv` alone. That check has been removed: without `year`/`university`/`school`/`degree`, it can't tell coincidental rounding apart from genuine duplicate programme-year records, so it isn't meaningful on its own. The real investigation happens in Section 8, once both datasets are merged and full cohort identity is available.

## 11. Final summary statistics (Employment Outcomes)

In [ ]:
ges_employment_outcomes_clean[outcome_cols].describe()

## 12. Merge and post-merge investigation

**Workflow for this section** (duplicates are deliberately *not* resolved until after investigation):

```
Merge using Key (validate one-to-one)
        ↓
Investigate repeated programme-year records
        ↓
Exclude unreliable 2012 records
        ↓
Resolve remaining logical duplicates (by data completeness)
        ↓
Final validation and save
```

### 12.1 Merge and validate

Merging on `Key` with `validate="one_to_one"` — this raises an error immediately if either side has a duplicate `Key`, which is a stronger guarantee than checking `.duplicated()` separately beforehand. `indicator=True` lets us confirm every row matched on both sides.

In [ ]:
ges_combined_clean = ges_degree_details_clean.merge(
    ges_employment_outcomes_clean,
    on="Key",
    how="inner",
    validate="one_to_one",
    indicator=True,
)

print("ges_degree_details_clean shape:", ges_degree_details_clean.shape)
print("ges_employment_outcomes_clean shape:", ges_employment_outcomes_clean.shape)
print("Merged shape (with indicator):", ges_combined_clean.shape)
print(ges_combined_clean["_merge"].value_counts())

In [ ]:
assert (ges_combined_clean["_merge"] == "both").all(), "Some rows did not match on both sides of the merge"
ges_combined_clean = ges_combined_clean.drop(columns="_merge")
print("Confirmed: every row matched on both sides. Shape after dropping indicator column:", ges_combined_clean.shape)

### 12.2 Investigate repeated programme-year records

Ben's Section 4 detected that many `(year, university, school, degree)` combinations repeat, but deliberately didn't act on it. Now that employment outcome data is attached, we can check whether the repeated copies actually agree with each other — which determines whether it's safe to just keep one, and if so, which one.

In [ ]:
outcome_cols = [
    "employment_rate_overall", "employment_rate_ft_perm",
    "basic_monthly_mean", "basic_monthly_median",
    "gross_monthly_mean", "gross_monthly_median",
    "gross_mthly_25_percentile", "gross_mthly_75_percentile",
]
cohort_cols = ["year", "university", "school", "degree"]

cohort_counts = ges_combined_clean.groupby(cohort_cols, dropna=False).size()
print("Unique cohorts:", cohort_counts.shape[0])
print("Total rows:", len(ges_combined_clean))
print(cohort_counts.value_counts().sort_index())

In [ ]:
def classify_group(rows):
    if rows.nunique(dropna=False).le(1).all():
        return "identical"
    all_null_row = rows.isna().all(axis=1)
    non_null_rows = rows.loc[~all_null_row]
    if all_null_row.any() and non_null_rows.nunique(dropna=False).le(1).all():
        return "null_vs_real"
    return "genuine_conflict"

dup_cohorts = cohort_counts[cohort_counts > 1].index
classification_counts = {"identical": 0, "null_vs_real": 0, "genuine_conflict": 0}
conflicting_groups = []

for cohort in dup_cohorts:
    year, uni, school, degree = cohort
    if pd.isna(school):
        mask = (
            (ges_combined_clean["year"] == year) & (ges_combined_clean["university"] == uni)
            & (ges_combined_clean["school"].isna()) & (ges_combined_clean["degree"] == degree)
        )
    else:
        mask = (
            (ges_combined_clean["year"] == year) & (ges_combined_clean["university"] == uni)
            & (ges_combined_clean["school"] == school) & (ges_combined_clean["degree"] == degree)
        )
    rows = ges_combined_clean.loc[mask, outcome_cols]
    label = classify_group(rows)
    classification_counts[label] += 1
    if label == "genuine_conflict":
        conflicting_groups.append(cohort)

print("Duplicate-group classification:")
for label, count in classification_counts.items():
    print(f"  {label}: {count}")
print()
print("Cohorts with genuinely conflicting (non-null) values:", conflicting_groups)

**Finding:** among the 1,263 repeated programme-year groups:

- 1,160 groups contain identical outcome values across every copy.
- 102 groups contain one missing copy while the remaining copies contain identical values.
- 1 group contains genuinely conflicting non-missing outcome values (`2012 / Nanyang Technological University / College of Engineering / Aerospace Engineering`).

Therefore, only one programme-year group required manual investigation — and it sits entirely inside `year == 2012`, which is investigated next.

### 12.3 Investigate the 2012 records specifically

The employment rate and salary columns for `year == 2012` looked suspicious earlier (Section 6.4: implausibly low salaries). Checking directly whether 2012 is genuine survey data, or something else.

In [ ]:
money_cols = [
    "basic_monthly_mean", "basic_monthly_median",
    "gross_monthly_mean", "gross_monthly_median",
    "gross_mthly_25_percentile", "gross_mthly_75_percentile",
]

y2012 = ges_combined_clean[ges_combined_clean["year"] == 2012]
y2013 = ges_combined_clean[ges_combined_clean["year"] == 2013]

progs_2012 = set(zip(y2012["university"], y2012["school"].fillna("<NA>"), y2012["degree"]))
progs_2013 = set(zip(y2013["university"], y2013["school"].fillna("<NA>"), y2013["degree"]))

print("2012 rows:", len(y2012), "| unique 2012 programmes:", len(progs_2012))
print("2012 programmes that also exist in 2013:", len(progs_2012 & progs_2013), "/", len(progs_2012))

**Note on the comparison below:** both `year == 2012` and `year == 2013` still contain their own repeated programme-year copies at this point (duplicates aren't resolved until Section 8.4), so comparing them directly would produce a many-to-many blow-up. To keep the comparison honest, we first build a single reference record per 2013 programme (the most complete copy), then compare each 2012 row against that one reference.

In [ ]:
programme_columns = ["university", "school", "degree"]

y2013_reference = y2013.copy()
y2013_reference["_completeness"] = y2013_reference[outcome_cols].notna().sum(axis=1)
y2013_reference = (
    y2013_reference
    .sort_values(["_completeness", "Key"], ascending=[False, True])
    .drop_duplicates(subset=programme_columns, keep="first")
    .drop(columns="_completeness")
)

compare = y2012.merge(
    y2013_reference, on=programme_columns, how="left",
    suffixes=("_2012", "_2013"), validate="many_to_one",
)

print("2012 rows compared:", len(compare))
print("Unique 2013 reference programmes:", len(y2013_reference))

In [ ]:
rate_columns = ["employment_rate_overall", "employment_rate_ft_perm"]

for column in rate_columns:
    value_2012 = compare[f"{column}_2012"]
    value_2013 = compare[f"{column}_2013"]
    both_missing = value_2012.isna() & value_2013.isna()
    same_value = value_2012.eq(value_2013) | both_missing
    print(f"{column}: {same_value.sum()}/{len(compare)} rows match")

In [ ]:
for column in money_cols:
    value_2012 = compare[f"{column}_2012"]
    value_2013 = compare[f"{column}_2013"]
    both_missing = value_2012.isna() & value_2013.isna()
    scaled_match = ((value_2012 * 10 - value_2013).abs() <= 1) | both_missing
    print(f"{column}: {scaled_match.sum()}/{len(compare)} rows match after ×10")

**Finding:** all 85 unique programmes reported under `year == 2012` also appear under `year == 2013`. Against a single clean 2013 reference record per programme, employment rates match for 85/86 rows and every salary column matches 2013 (after undoing the ×10 scale) for 84-85/86 rows — the sole mismatch each time being the same Aerospace Engineering conflict from Section 8.2. In other words, `year == 2012` is not independent survey data — it's a near-total duplicate of `year == 2013`, mislabeled under a different year and with a scaling error layered on top. There's no reliable way to "correct" data that was never really 2012's own in the first place.

**Decision:** exclude `year == 2012` from the final dataset entirely, rather than attempting to rescale it.

In [ ]:
rows_before = len(ges_combined_clean)

ges_combined_clean = (
    ges_combined_clean
    .loc[ges_combined_clean["year"].between(2013, 2022)]
    .copy()
    .reset_index(drop=True)
)

print(f"Rows before excluding 2012: {rows_before}")
print(f"Rows after excluding 2012: {len(ges_combined_clean)}")
print(f"Rows removed: {rows_before - len(ges_combined_clean)}")

### 12.4 Resolve remaining logical duplicates (by completeness, not just "first")

With `year == 2012` gone, Section 8.2 showed the only remaining kind of duplication is "one copy missing, others identical" — never a genuine conflict. Rather than assuming the first `Key` happens to be the complete one, we resolve this explicitly: for each cohort, keep whichever copy has the most non-missing outcome values, breaking ties by the lowest `Key`.

In [ ]:
ges_combined_clean["_completeness"] = ges_combined_clean[outcome_cols].notna().sum(axis=1)

ges_combined_clean = (
    ges_combined_clean
    .sort_values(["_completeness", "Key"], ascending=[False, True])
    .drop_duplicates(subset=cohort_cols, keep="first")
    .drop(columns="_completeness")
    .sort_values("Key")
    .reset_index(drop=True)
)

print("Shape after resolving duplicates by completeness:", ges_combined_clean.shape)

### 12.5 Final salary outlier check (after full cleaning)

Section 6.3 already checked outliers with a boxplot and IQR, but that was on `ges_employment_outcomes_clean` before the merge, before excluding the corrupted `year == 2012` rows, and before resolving duplicate cohorts. That earlier check was useful for spotting that something was off (it's part of what led to investigating 2012 in the first place), but it isn't the right basis for reporting final outlier statistics, since the corrupted/duplicated rows inflate the counts. This repeats the same check on `ges_combined_clean`, the actual final dataset.

In [ ]:
final_salary_columns = ["basic_monthly_mean", "basic_monthly_median", "gross_monthly_mean", "gross_monthly_median"]

plt.figure(figsize=(10, 6))
sns.boxplot(data=ges_combined_clean[final_salary_columns])
plt.title("Distribution of Monthly Salary Figures After Final Cleaning")
plt.ylabel("S$ per month")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
for column in final_salary_columns:
    lower, upper, count = iqr_outlier_count(ges_combined_clean[column].dropna())
    print(f"{column}: bounds=({lower:.0f}, {upper:.0f}), outliers={count}")

**Decision:** these outliers are kept, not removed — legitimately high- and low-earning cohorts matter for the research question. What's changed is that these counts now reflect the real, final dataset rather than one still containing the corrupted 2012 rows and unresolved duplicates.

### 12.6 Final quality assurance

In [ ]:
print("FINAL COMBINED DATASET — QUALITY SUMMARY")
print("-" * 45)
print("Shape:", ges_combined_clean.shape)
print()
print("Dtypes:")
print(ges_combined_clean.dtypes)
print()
print("Missing values:")
print(ges_combined_clean.isna().sum())
print()
print("Exact duplicate rows:", ges_combined_clean.duplicated().sum())
print("Duplicate Key values:", ges_combined_clean["Key"].duplicated().sum())
print("Duplicate (year, university, school, degree) cohorts:",
      ges_combined_clean.duplicated(subset=cohort_cols).sum())
print()
print("Year range:", ges_combined_clean["year"].min(), "-", ges_combined_clean["year"].max())

In [ ]:
ges_combined_clean[money_cols + ["employment_rate_overall", "employment_rate_ft_perm"]].describe()

### 12.7 Final validation: missing outcomes occur only as complete blocks

A cohort's employment outcome data should either be fully reported or fully suppressed (small-cohort privacy rules) — never partially present. Checking this holds for every row before exporting.

In [ ]:
outcome_values_present = ges_combined_clean[outcome_cols].notna().sum(axis=1)

fully_missing_outcomes = (outcome_values_present == 0).sum()
partially_missing_outcomes = ((outcome_values_present > 0) & (outcome_values_present < len(outcome_cols))).sum()
complete_outcomes = (outcome_values_present == len(outcome_cols)).sum()

print("Rows with complete outcomes:", complete_outcomes)
print("Rows with all outcomes missing:", fully_missing_outcomes)
print("Rows with partially missing outcomes:", partially_missing_outcomes)

assert partially_missing_outcomes == 0

### 12.8 Final sanity checks

A last set of range/validity assertions before saving — employment rates must be genuine percentages, and no salary figure should be negative.

In [ ]:
assert ges_combined_clean["employment_rate_overall"].dropna().between(0, 100).all()
assert ges_combined_clean["employment_rate_ft_perm"].dropna().between(0, 100).all()
assert not (ges_combined_clean[money_cols] < 0).any().any()

print("All sanity checks passed: employment rates in [0, 100], no negative salary values.")

### 12.9 Save the final combined dataset

In [ ]:
ges_combined_clean.to_csv("ges_combined_cleaned.csv", index=False)
print("Saved: ges_combined_cleaned.csv")
print("Final shape:", ges_combined_clean.shape)

## Summary of cleaning and merge steps

**Degree Details (Ben) — detection only, resolution deferred to Section 8:**
1. Standardised placeholder text values to real missing values in `university`/`school`/`degree`.
2. Stripped whitespace from text columns.
3. Detected and removed 1 row with an invalid `year` value (the embedded-header corruption, same `Key` as the one found independently in `ges_employment_outcomes.csv`), resetting the index afterward.
4. Converted `year` to a proper nullable integer dtype.
5. Corrected inconsistent capitalisation in `school` and `degree` names.
6. **Detected** (but deliberately did not remove) duplicated `(year, university, school, degree)` records — 3,787 excess records retained for post-merge investigation. `ges_degree_details_clean` stays at `(5134, 5)`.

**Employment Outcomes (Tahmin):**
7. Removed 35 fully empty `Unnamed` columns and 4 duplicate `.1`-suffixed columns.
8. Replaced the `'na'` string placeholder with proper `NaN` and converted all 8 outcome columns to numeric dtype.
9. Checked logical consistency (basic vs gross pay, percentile ordering) and did a preliminary IQR outlier check (later superseded by Section 8.5, once the data is actually final).
10. Removed the outcome-only duplicate check — not meaningful without cohort identity.

**Combined (Section 8) — the order matters:**
11. Merged both cleaned datasets on `Key` with `validate="one_to_one"` and confirmed every row matched on both sides.
12. Investigated the repeated programme-year records *with* employment data attached: 1,160 groups are exact repeats, 102 have one all-missing copy, and exactly 1 has a genuine conflict — confined entirely to `year == 2012`.
13. Investigated `year == 2012` directly against a single clean 2013 reference record per programme (avoiding a many-to-many blow-up): 85/85 programmes also appear in `year == 2013`, with matching employment rates and salaries (after undoing a 10x scale error) for all but the one conflicting programme. Concluded 2012 is not independent survey data and **excluded it**, rather than rescaling it.
14. Resolved the remaining duplicates by keeping the most *complete* copy of each cohort (most non-missing outcome values, tie-broken by lowest `Key`) — not just an arbitrary "first" row.
15. Re-ran the salary outlier check (boxplot + IQR) on the final `ges_combined_clean`, since the Section 9/6.3 numbers were inflated by rows that no longer belong in the dataset.
16. Ran a final QA pass (dtypes, nulls, duplicate rows/Keys/cohorts).
17. Verified missing outcome values only ever occur as fully-missing rows, never partially missing.
18. Ran final sanity assertions: employment rates in [0, 100], no negative salary values.
19. Saved `ges_combined_cleaned.csv`.

**Final dataset:** 1,262 rows, one row per genuine programme-year cohort, spanning `year` 2013–2022.